# 02_03 · Preprocesado — Dataset CNC Industrial (KIT Karlsruhe)

**Entrada**: series temporales brutas a 500 Hz en `data/raw/kit_extracted/`
**Salida**: `industrial_features_v2.csv` — un vector de features por experimento

Pipeline de preprocesado:
1. Carga de las 18 series temporales
2. Feature engineering: dominio temporal (10 estadísticos) + dominio frecuencia (3 bandas energéticas)
3. Etiquetas binarias y multi-clase
4. StandardScaler
5. Guardado

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
%matplotlib inline

KIT_DIR   = os.path.join('..', 'data', 'raw', 'kit_extracted')
PROC_DIR  = os.path.join('..', 'data', 'processed')
os.makedirs(PROC_DIR, exist_ok=True)

KEY_SIGNALS = [
    'TORQUE|1','TORQUE|2','TORQUE|3','TORQUE|6',
    'CURRENT|1','CURRENT|2','CURRENT|3','CURRENT|6',
    'POWER|1','POWER|2','POWER|3','POWER|6',
    'LOAD|1','LOAD|2','LOAD|3','LOAD|6',
]

# Etiquetas multi-clase — 33 experimentos completos
FAULT_TYPE = {
    # 0 — Normal
    'IM-01F': 0, 'IM-01R': 0,
    'IMP-01': 0, 'IMP-02': 0, 'IMP-03': 0, 'IMP-04': 0,
    'IMP-05': 0, 'IMP-06': 0, 'IMP-07': 0, 'IMP-08': 0,
    'IMP-09': 0, 'IMP-10': 0, 'IMP-11': 0, 'IMP-12': 0, 'IMP-BASE': 0,
    'TF-01': 0, 'TF-02': 0, 'TF-03': 0,
    # 1 — Tool Wear
    'IM-01R-A01': 1, 'IM-01R-A02': 1, 'IM-01R-A03': 1,
    'IM-01R-A04': 1, 'IM-01R-A05': 1,
    # 2 — Chatter
    'IM-02F-A01': 2,
    # 3 — Process Anomaly (overload, feedrate, built-up edge, stock)
    'IM-01F-A01': 3, 'TF-01-A01': 3, 'TF-02-A01': 3,
    'TF-03-A01': 3, 'TF-03-A02': 3,
    # 4 — Workpiece Defect (cavity, crack, chipped edge)
    'IMP-01-A01': 4, 'IMP-01-A02': 4, 'IMP-01-A03': 4, 'IMP-01-A04': 4,
}
FAULT_NAMES  = {0:'Normal', 1:'Tool Wear', 2:'Chatter',
                3:'Process Anomaly', 4:'Workpiece Defect'}
FAULT_COLORS = {0:'#3498db', 1:'#e74c3c', 2:'#e67e22', 3:'#9b59b6', 4:'#2ecc71'}
BINARY = {t: int(l > 0) for t, l in FAULT_TYPE.items()}

def load_kit(trial):
    path = os.path.join(KIT_DIR, f'{trial}_hfdata.csv')
    if not os.path.exists(path):
        return None
    df = pd.read_csv(path)
    df['trial']      = trial
    df['fault_type'] = FAULT_TYPE.get(trial, -1)
    df['anomaly']    = BINARY.get(trial, -1)
    return df

TRIALS = sorted(FAULT_TYPE.keys())
n_norm = sum(v==0 for v in FAULT_TYPE.values())
n_anom = sum(v>0  for v in FAULT_TYPE.values())
print(f'Trials: {len(TRIALS)} ({n_norm} normales, {n_anom} anomalias)')
print(f'Por tipo: { {FAULT_NAMES[k]: sum(v==k for v in FAULT_TYPE.values()) for k in FAULT_NAMES} }')


Trials: 33 (18 normales, 15 anomalias)
Por tipo: {'Normal': 18, 'Tool Wear': 5, 'Chatter': 1, 'Process Anomaly': 5, 'Workpiece Defect': 4}


## 1. Carga de las series temporales

Cargamos los 18 experimentos y verificamos que las señales clave estén disponibles.
Cada experimento tiene entre 82.000 y 606.000 filas (duración 2.7–20 minutos a 500 Hz).

In [2]:
dfs = {}
for t in TRIALS:
    df = load_kit(t)
    if df is not None:
        dfs[t] = df
        print(f'{t:<20} {FAULT_NAMES[FAULT_TYPE[t]]:<18} {len(df):>8,} filas')
    else:
        print(f'  ⚠ No encontrado: {t}')

print(f'\n{len(dfs)} experimentos cargados')

IM-01F               Normal              308,454 filas
IM-01F-A01           Process Anomaly     306,867 filas
IM-01R               Normal              610,343 filas
IM-01R-A01           Tool Wear           604,462 filas
IM-01R-A02           Tool Wear           606,127 filas
IM-01R-A03           Tool Wear           604,182 filas
IM-01R-A04           Tool Wear           608,929 filas
IM-01R-A05           Tool Wear           604,489 filas
IM-02F-A01           Chatter              92,362 filas
IMP-01               Normal              172,052 filas
IMP-01-A01           Workpiece Defect    173,522 filas
IMP-01-A02           Workpiece Defect    174,954 filas
IMP-01-A03           Workpiece Defect    174,482 filas
IMP-01-A04           Workpiece Defect    173,821 filas
IMP-02               Normal              176,346 filas
IMP-03               Normal              148,169 filas
IMP-04               Normal              132,360 filas
IMP-05               Normal              156,752 filas
IMP-06    

## 2. Feature engineering — dominio temporal

Para cada señal calculamos 10 estadísticos que capturan distintos aspectos de la distribución:

| Feature | Qué captura | Relevante para |
|---|---|---|
| `mean` | Nivel medio de operación | Overload (mayor TORQUE medio) |
| `std` | Variabilidad general | Chatter (mayor STD) |
| `rms` | Energía efectiva (√(∑x²/n)) | Tool Wear (mayor energía) |
| `max` | Pico máximo absoluto | Workpiece Defect (pico impulsivo) |
| `peak_to_peak` | Rango dinámico | Chatter (oscilación) |
| `kurtosis` | Impulsividad de la distribución | Defectos y chatter |
| `skewness` | Asimetría | Tool Wear (distribución sesgada) |
| `crest_factor` | max / rms — impulsividad normalizada | Defectos puntuales |
| `zcr` | Tasa de cruces por cero | Chatter (alta frecuencia de oscilación) |
| `mad` | Mediana de desviaciones absolutas (robusto a outliers) | Señal robusta del nivel base |

`RMS` es la métrica estándar en monitorización de maquinaria — es mucho más informativa
que la media porque captura tanto el nivel como la energía de las fluctuaciones.

In [3]:
from scipy.stats import kurtosis, skew

def time_features(vals):
    """10 estadísticos en el dominio temporal sobre un array 1D."""
    vals = vals[~np.isnan(vals)]
    if len(vals) < 10:
        return {k: np.nan for k in ['mean','std','rms','max','p2p','kurt','skew','crest','zcr','mad']}
    rms = np.sqrt(np.mean(vals**2))
    mx  = np.max(np.abs(vals))
    return {
        'mean':  np.mean(vals),
        'std':   np.std(vals),
        'rms':   rms,
        'max':   mx,
        'p2p':   mx - np.min(np.abs(vals)),
        'kurt':  kurtosis(vals),
        'skew':  skew(vals),
        'crest': mx / rms if rms > 1e-10 else 0,
        'zcr':   np.mean(np.diff(np.sign(vals - vals.mean())) != 0),
        'mad':   np.median(np.abs(vals - np.median(vals))),
    }

# Test rápido
df_test = dfs[list(dfs.keys())[0]]
if 'TORQUE|6' in df_test.columns:
    feats = time_features(df_test['TORQUE|6'].values)
    print('Features temporales calculadas:', list(feats.keys()))
    for k,v in feats.items():
        print(f'  {k:<8} {v:.4f}')

Features temporales calculadas: ['mean', 'std', 'rms', 'max', 'p2p', 'kurt', 'skew', 'crest', 'zcr', 'mad']
  mean     -0.8337
  std      2.5431
  rms      2.6763
  max      85.3165
  p2p      85.3165
  kurt     847.1981
  skew     -7.7751
  crest    31.8783
  zcr      0.1533
  mad      0.1061


## 3. Feature engineering — dominio de frecuencia

La FFT transforma la señal temporal en su contenido espectral.
Dividimos el espectro (0–250 Hz) en 3 bandas de energía:

| Banda | Rango | Qué captura |
|---|---|---|
| Baja | 0–25 Hz | Movimientos lentos de los ejes, variación de profundidad de corte |
| Media | 25–100 Hz | Vibraciones de baja frecuencia, resonancias estructurales leves |
| Alta | 100–250 Hz | Chatter (resonancia del husillo) y ruido de mecanizado de alta frecuencia |

Además calculamos la **frecuencia dominante**: el pico espectral principal.
En presencia de chatter, habrá un pico claro en la banda alta.

In [4]:
from scipy.fft import rfft, rfftfreq

FS = 500  # Hz

def freq_features(vals):
    """Energía en 3 bandas espectrales + frecuencia dominante."""
    vals = vals[~np.isnan(vals)]
    if len(vals) < 100:
        return {'e_low':np.nan,'e_mid':np.nan,'e_high':np.nan,'f_dom':np.nan}
    seg = vals - vals.mean()
    yf  = np.abs(rfft(seg))
    xf  = rfftfreq(len(seg), 1/FS)
    total = yf.sum() + 1e-10
    return {
        'e_low':  yf[(xf < 25)].sum() / total,
        'e_mid':  yf[(xf >= 25) & (xf < 100)].sum() / total,
        'e_high': yf[(xf >= 100)].sum() / total,
        'f_dom':  xf[np.argmax(yf)],
    }

# Test
if 'TORQUE|6' in df_test.columns:
    ff = freq_features(df_test['TORQUE|6'].values)
    print('Features frecuenciales:', ff)

Features frecuenciales: {'e_low': np.float64(0.9355965024781927), 'e_mid': np.float64(0.049359777035197165), 'e_high': np.float64(0.015043720486609997), 'f_dom': np.float64(0.05835554085860452)}


## 4. Construcción del dataset de features

Para cada uno de los 18 experimentos calculamos todos los estadísticos
sobre todas las señales disponibles → matriz de features `(18 experimentos × n_features)`.

Total features esperadas: `5 tipos de señal × 4 ejes × (10 temporales + 4 frecuenciales) = 280 features`
(más las que están disponibles en cada experimento).

In [7]:
rows = []
for t, df in dfs.items():
    row = {
        'trial':      t,
        'fault_type': FAULT_TYPE[t],
        'fault_name': FAULT_NAMES[FAULT_TYPE[t]],
        'anomaly':    BINARY[t],
        'n_samples':  len(df),
    }
    for sig in KEY_SIGNALS:
        if sig not in df.columns:
            continue
        vals = df[sig].values.astype(float)
        tf = time_features(vals)
        ff = freq_features(vals)
        prefix = sig.replace('|','_ax')
        for k, v in tf.items():
            row[f'{prefix}_{k}'] = v
        for k, v in ff.items():
            row[f'{prefix}_{k}'] = v
    rows.append(row)

features_df = pd.DataFrame(rows)
feat_cols = [c for c in features_df.columns
             if c not in ('trial','fault_type','fault_name','anomaly','n_samples')]

print(f'Experimentos: {len(features_df)}')
print(f'Features totales: {len(feat_cols)}')
print(f'NaN: {features_df[feat_cols].isnull().sum().sum()}')
print()
print(features_df[['trial','fault_name','n_samples']].head(5))

Experimentos: 33
Features totales: 224
NaN: 112

        trial       fault_name  n_samples
0      IM-01F           Normal     308454
1  IM-01F-A01  Process Anomaly     306867
2      IM-01R           Normal     610343
3  IM-01R-A01        Tool Wear     604462
4  IM-01R-A02        Tool Wear     606127


## 5. Escalado y guardado

Aplicamos `StandardScaler` para normalizar las features — necesario porque tienen
escalas muy distintas (TORQUE en Nm vs CURRENT en A vs POWER en W).

El scaler se ajusta sobre **todos los experimentos** porque tenemos muy pocos datos (18).
En producción se ajustaría solo sobre los normales.

In [6]:
import pickle
from sklearn.preprocessing import StandardScaler

# Rellenar NaN con 0 (señal no disponible en ese experimento)
features_df[feat_cols] = features_df[feat_cols].fillna(0)

# Guardar CSV completo
features_df.to_csv(os.path.join(PROC_DIR, 'industrial_features_v2.csv'), index=False)
print(f'Guardado: industrial_features_v2.csv ({features_df.shape})')

# StandardScaler sobre todas las muestras
X = features_df[feat_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

with open(os.path.join(PROC_DIR, 'industrial_scaler.pkl'), 'wb') as f:
    pickle.dump({'scaler': scaler, 'feat_cols': feat_cols}, f)
print(f'Scaler guardado: industrial_scaler.pkl')
print(f'\nResumen de features (primeras 8):')
pd.DataFrame(X_scaled, columns=feat_cols, index=features_df['trial']).iloc[:, :8].round(2)

Guardado: industrial_features_v2.csv ((33, 229))
Scaler guardado: industrial_scaler.pkl

Resumen de features (primeras 8):


,TORQUE_ax1_mean,TORQUE_ax1_std,TORQUE_ax1_rms,TORQUE_ax1_max,TORQUE_ax1_p2p,TORQUE_ax1_kurt,TORQUE_ax1_skew,TORQUE_ax1_crest
trial,,,,,,,,
IM-01F,0.28,-0.66,-0.68,0.32,0.32,0.67,0.44,1.33
IM-01F-A01,0.25,-0.67,-0.69,0.35,0.35,0.74,0.48,1.38
IM-01R,-1.41,0.91,0.94,0.71,0.71,0.44,1.46,-0.80
IM-01R-A01,-1.32,1.04,1.07,0.72,0.72,0.19,1.42,-0.94
IM-01R-A02,-1.29,1.07,1.10,0.73,0.73,0.11,1.39,-0.96
IM-01R-A03,-1.47,1.09,1.13,0.72,0.72,-0.02,1.27,-1.00
IM-01R-A04,-1.50,1.17,1.20,0.70,0.70,-0.17,1.29,-1.09
IM-01R-A05,-1.57,1.18,1.21,0.70,0.70,-0.19,1.14,-1.10
IM-02F-A01,0.43,1.66,1.66,0.41,0.41,-0.42,-1.13,-1.73


## 6. Conclusiones — Preprocesado KIT

- **280 features** por experimento (10 temporales + 4 frecuenciales × 5 señales × 4 ejes)
- **RMS y kurtosis** son las métricas más informativas para detección de anomalías en maquinado
- **Energía en banda alta (100–250 Hz)** discrimina chatter del resto — será clave en el modelo
- **POWER|6** contribuye features de variabilidad clave para chatter ni SPARK
- Listo para clasificación en `03_03_Modelos_Industrial.ipynb`